In [1]:
import pylbo as pb
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
case = "HELALL"

In [ ]:
os.system("rm "+case+"/output/*")
os.system("rm "+case+"/parfiles/*")

k3Val = np.linspace(0.5, 8.0 , 50)

counter=0

for val in k3Val:

    config = {
        "number_of_runs": 50,
        "gridpoints": 101,
        "equilibrium_type": "user_defined",
        "parameters": {

            "V": 1.63,
            "cte_rho0": 1.0,
            "cte_p0": 1.0,
            "Bz0": 0.25,
            "rj": 1.0,
            "rc": np.linspace(0.25, 4.0, 50),
            "k2": -1.0,
            "k3": val
        },
        "write_eigenfunctions": True,
        "basename_datfile": "khcd_spectrum_"+case,
        "write_derived_eigenfunctions": True,
        "output_folder": case+"/output",
        "solver": "QR-cholesky",
        "use_defaults": False
    }

    pb.generate_parfiles(config, basename="khcd_spectrum_"+case+str(counter), output_dir=case)
    counter += 1

parfiles = sorted(Path(case+"/parfiles").glob("*.par")) 

In [ ]:
pb.run_legolas(parfiles, executable="legolas", nb_cpus=8)

2026-03-25 22:59:05,467 - |INFO    | initialising runner, using executable /home/areen/Documents/Projects/astrophysical-jet-stability/jetFolder/legolas


running legolas [8 CPUS]:   0%|          | 0/50 [00:00<?, ?/s]

 
     __       ________  ________   _______   __          ___     __________
 
    |  |     |   ____ \|   ____ \ /   _   \ |  |        /   \   |   ______ \
    |  |     |  |    \/|  |    \/|   / \   ||  |       /  _  \  |  |      \/
     __       ________  ________   _______   __          ___     __________
    |  |     |  |__    |  |      |  |   |  ||  |      /  / \  \ |  \_______
    |  |     |   ____ \|   ____ \ /   _   \ |  |        /   \   |   ______ \
    |  |     |   __/   |  | ____ |  |   |  ||  |     /  /   \  \\_______   \
    |  |     |  |    \/|  |    \/|   / \   ||  |       /  _  \  |  |      \/
    |  |     |  |      |  | \_  ||  |   |  ||  |    /  /     \  \       |  |
    |  |     |  |__    |  |      |  |   |  ||  |      /  / \  \ |  \_______
    |  |_____|  |____/\|  |___| ||   \_/   ||  |___/  /  /\___\  \      |  |
    |  |     |   __/   |  | ____ |  |   |  ||  |     /  /   \  \\_______   \
    |_______/|________/|________| \_______/ |________/   \_______/      |  |

In [ ]:
datfiles = sorted(Path(case+"/output/").glob("*khcd_spectrum_"+case+"*.dat"))

series = pb.load_series(datfiles)

In [ ]:
k3 = []
growthRate = []
frequencies = []
eigenfunctions = []

for spectra in series:

    eigenVal = spectra.eigenvalues

    eigenPos = []

    for eigenId, evs in enumerate(eigenVal):
        
        omega = evs.real
        gamma = evs.imag

        if(gamma>1e-2):
            growthRate.append(gamma)
            frequencies.append(omega)

            k3.append(spectra.parameters["k3"])

    eigenPos.append(eigenId)

    efs = spectra.get_eigenfunctions(ev_idxs=eigenPos)

    eigenfunctions.append(efs)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(k3, growthRate, s=4, color="gray",  label="all modes", alpha=0.5)

ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("$k_3$")
ax.set_ylabel(r"Growth rate ")
ax.set_title("KH vs CD mode separation")
ax.legend()
plt.tight_layout()
plt.show()